In [1]:
%cd /glade/derecho/scratch/lizhili/m2l8/sr_model_code/UPSR_M2L8

/glade/derecho/scratch/lizhili/m2l8/sr_model_code/UPSR_M2L8


/glade/derecho/scratch/lizhili/UPSR_env/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt
import torch.nn.functional as F
import tensorflow as tf
import numpy as np
import torch.nn as nn

2026-03-08 10:40:17.914282: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Use only GPU 1

In [4]:
from basicsr.utils.options import ordered_yaml
import yaml
file_load = '/glade/derecho/scratch/lizhili/UPSR/options/000_UPSR_RealSR_x4.yml'
with open(file_load) as f:
    opt = yaml.load(f, Loader=ordered_yaml()[0])

/glade/derecho/scratch/lizhili/UPSR_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/glade/derecho/scratch/lizhili/UPSR_env/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [5]:
opt['is_train'] = True
opt['scale']=4
opt['dist']=False
opt['rank'] = 0
opt['network_mse']['params']['in_chans']=7
opt['network_mse']['ckpt']['strict_load_mse']=False
opt['network_mse']['params']['upscale']=4
opt['network_g']['params']['out_channels']=7
opt['network_g']['params']['in_channels']=112

In [6]:
from basicsr.models import build_model
model = build_model(opt)

/glade/derecho/scratch/lizhili/UPSR_env/lib/python3.10/site-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3609.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/glade/derecho/scratch/lizhili/m2l8/sr_model_code/UPSR_M2L8/basicsr/models/upsr_real_model.py:101: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via 

sqrt_etas:  [0.03162278 0.43001096 0.63390775 0.81660213 0.99      ]
Loading pretrained model LPIPS from /glade/u/home/lizhili/.cache/torch/hub/pyiqa/LPIPS_v0.1_vgg-a78928a0.pth
Loading pretrained model LPIPS from /glade/u/home/lizhili/.cache/torch/hub/pyiqa/LPIPS_v0.1_alex-df73285e.pth
Loading pretrained model MUSIQ from /glade/u/home/lizhili/.cache/torch/hub/pyiqa/musiq_koniq_ckpt-e95806b9.pth


In [7]:
def input_pipeline(filename, batch_size, is_shuffle=True, is_train=True, is_repeat=True):
    feature_description = {
        'lres': tf.io.FixedLenFeature([60*60*7], dtype=tf.int64),
        'hres': tf.io.FixedLenFeature([1000*1000*7], dtype=tf.int64),
    }

    def _parse_function(example_proto):
        feature_dict = tf.io.parse_single_example(example_proto, feature_description)
        lres_img = tf.reshape(feature_dict['lres'], [7, 60, 60])
        hres_img = tf.reshape(feature_dict['hres'], [7, 1000, 1000])

        return lres_img, hres_img

    def _augment_function(lres_img, hres_img):
    # Transpose to [H, W, C]
        lres_img = tf.transpose(lres_img, [1, 2, 0])   # [60, 60, 12]
        hres_img = tf.transpose(hres_img, [1, 2, 0])   # [1000, 1000, 4]

        # Randomly choose 0, 90, 180, or 270 degrees
        k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)

        # Apply rotation
        lres_img = tf.image.rot90(lres_img, k=k)
        hres_img = tf.image.rot90(hres_img, k=k)

        # Transpose back to [C, H, W]
        lres_img = tf.transpose(lres_img, [2, 0, 1])   # [12, 60, 60]
        hres_img = tf.transpose(hres_img, [2, 0, 1])   # [4, 1000, 1000]

        return lres_img, hres_img

    dataset = tf.data.TFRecordDataset(filename)
    if is_repeat:
        dataset = dataset.repeat()
    dataset = dataset.map(_parse_function)
    if is_train:
        dataset = dataset.map(_augment_function)
    if is_shuffle:
        dataset = dataset.shuffle(buffer_size=100)
    batch = dataset.batch(batch_size=batch_size)
    return batch

In [8]:
def random_crop_lr_hr(img_lr, img_hr, lr_crop_size):
    """
    Random aligned crop for super-resolution pairs.

    img_lr: Tensor [B, C, 64, 64]
    img_hr: Tensor [B, C, 1024, 1024]
    lr_crop_size: int (e.g., 32)

    Returns:
        lr_crop: [B, C, lr_crop_size, lr_crop_size]
        hr_crop: [B, C, lr_crop_size*16, lr_crop_size*16]
    """
    scale = img_hr.shape[-1] // img_lr.shape[-1]  # 16

    _, _, H_lr, W_lr = img_lr.shape
    assert H_lr >= lr_crop_size and W_lr >= lr_crop_size

    top_lr = torch.randint(0, H_lr - lr_crop_size + 1, (1,)).item()
    left_lr = torch.randint(0, W_lr - lr_crop_size + 1, (1,)).item()

    top_hr = top_lr * scale
    left_hr = left_lr * scale

    lr_crop = img_lr[:, :, top_lr:top_lr+lr_crop_size,
                             left_lr:left_lr+lr_crop_size]

    hr_crop = img_hr[:, :, top_hr:top_hr+lr_crop_size*scale,
                             left_hr:left_hr+lr_crop_size*scale]

    return lr_crop, hr_crop

In [9]:
ckpt = torch.load('/glade/derecho/scratch/lizhili/m2l8/upsr_m2l8_x4.pth', map_location='cuda')

model.get_bare_model(model.net_g).load_state_dict(ckpt['net_g'], strict=True)
model.get_bare_model(model.net_mse).load_state_dict(ckpt['net_mse'], strict=True)
# model.get_bare_model(model.net_g_ema).load_state_dict(model.get_bare_model(model.net_g).state_dict(),strict=True)

if 'opt_g' in ckpt:
    model.optimizer_g.load_state_dict(ckpt['opt_g'])

if 'net_g_ema' in ckpt and hasattr(model, 'net_g_ema'):
    model.get_bare_model(model.net_g_ema).load_state_dict(ckpt['net_g_ema'], strict=True)

/glade/derecho/scratch/lizhili/tmp/ipykernel_11593/2293513743.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load('/glade/derecho/scratch/lizhili/m2l8/upsr

In [ ]:
import torch.nn.functional as F
import tensorflow as tf
import torch

filenames = ['/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_0.tfrecords',
             '/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_1.tfrecords',
             '/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_2.tfrecords',
             '/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_3.tfrecords',
             '/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_4.tfrecords']
ds = input_pipeline(filenames, batch_size=16, is_shuffle=True, is_train=True, is_repeat=True)
for epoch in range(1):
    print(f'Epoch {epoch}')

    for step, (lr, hr) in enumerate(ds):
        lr = torch.from_numpy(lr.numpy().astype('float32')).to('cuda')*0.0001
        hr = torch.from_numpy(hr.numpy().astype('float32')).to('cuda')*0.0000275-0.2

        lr = F.interpolate(lr, size=(64, 64), mode='bilinear', align_corners=False)
        hr = F.interpolate(hr, size=(256, 256), mode='bilinear', align_corners=False)

        lr = (lr-0.5)/0.5
        hr = (hr-0.5)/0.5

        current_iter = step+11001
        model.update_learning_rate(current_iter, warmup_iter=opt['train'].get('warmup_iter', -1))
        model.feed_data({'lq': lr, 'gt': hr}, training=False)
        model.optimize_parameters(current_iter)

        if step % 500 == 0:
            print(f'Step {step}, Loss: {model.get_current_log()}')
            # break
            ckpt = {
                'iter': current_iter,
                'net_g': model.get_bare_model(model.net_g).state_dict(),
                'net_mse': model.get_bare_model(model.net_mse).state_dict(),
                'opt_g': model.optimizer_g.state_dict(),  # optional (resume training)
            }

            if hasattr(model, 'net_g_ema'):
                ckpt['net_g_ema'] = model.get_bare_model(model.net_g_ema).state_dict()

            if getattr(model, 'amp_scaler', None) is not None:
                ckpt['amp_scaler'] = model.amp_scaler.state_dict()  # optional

            torch.save(ckpt, '/glade/derecho/scratch/lizhili/m2l8/upsr_m2l8_x4.pth')

        if step > 100000:
            break

Epoch 0


2026-02-26 00:37:22.157402: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:453] ShuffleDatasetV3:29: Filling up shuffle buffer (this may take a while): 31 of 100
2026-02-26 00:37:42.110985: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:453] ShuffleDatasetV3:29: Filling up shuffle buffer (this may take a while): 92 of 100
2026-02-26 00:37:44.741395: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:483] Shuffle buffer filled.
/glade/derecho/scratch/lizhili/UPSR_env/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:216: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


Step 0, Loss: OrderedDict([('l_pix', 0.027788430452346802), ('l_lpips', 0.024078369140625)])
Step 500, Loss: OrderedDict([('l_pix', 0.060046326369047165), ('l_lpips', 0.0537109375)])
Step 1000, Loss: OrderedDict([('l_pix', 0.05152131989598274), ('l_lpips', 0.04559326171875)])
Step 1500, Loss: OrderedDict([('l_pix', 0.07463099807500839), ('l_lpips', 0.06817626953125)])
Step 2000, Loss: OrderedDict([('l_pix', 0.03673572093248367), ('l_lpips', 0.032745361328125)])
Step 2500, Loss: OrderedDict([('l_pix', 0.034238070249557495), ('l_lpips', 0.029876708984375)])
Step 3000, Loss: OrderedDict([('l_pix', 0.044119372963905334), ('l_lpips', 0.03802490234375)])
Step 3500, Loss: OrderedDict([('l_pix', 0.04158893600106239), ('l_lpips', 0.035247802734375)])
Step 4000, Loss: OrderedDict([('l_pix', 0.05384160950779915), ('l_lpips', 0.047515869140625)])
Step 4500, Loss: OrderedDict([('l_pix', 0.03963000699877739), ('l_lpips', 0.03521728515625)])
Step 5000, Loss: OrderedDict([('l_pix', 0.05158940702676773

In [10]:
def downstream_run(lres_size ,
                    hres_size ,
                    hres_size_4x ,
                    label_size ,
                    num_sample,
                    num_training ,
                    finetune_tfrecords,
                    finetuned_model,
                    output_tfrecords,
                    model):
    # model.net_g = load_matched_weights(model.net_g, '/glade/derecho/scratch/lizhili/s2naip/BiDiff_x16_epoch_weights_finetune.pth')
    ckpt = torch.load('/glade/derecho/scratch/lizhili/m2l8/upsr_m2l8_x4.pth', map_location='cuda')

    model.get_bare_model(model.net_g).load_state_dict(ckpt['net_g'], strict=True)
    model.get_bare_model(model.net_mse).load_state_dict(ckpt['net_mse'], strict=True)
    
    if 'opt_g' in ckpt:
        model.optimizer_g.load_state_dict(ckpt['opt_g'])
    
    if 'net_g_ema' in ckpt and hasattr(model, 'net_g_ema'):
        model.get_bare_model(model.net_g_ema).load_state_dict(ckpt['net_g_ema'], strict=True)

    num_test = num_sample-num_training

    def input_pipeline_downstream_sr(filename, batch_size, skip, take, is_shuffle=True, is_train=True, is_repeat=True):
        feature_description = {
            'lres': tf.io.FixedLenFeature([7*lres_size*lres_size], dtype=tf.int64),
            'hres': tf.io.FixedLenFeature([7*hres_size*hres_size], dtype=tf.int64),
            'label': tf.io.FixedLenFeature([label_size*label_size], dtype=tf.int64)
        }

        @tf.function
        def _parse_function(example_proto):
            feature_dict = tf.io.parse_single_example(example_proto, feature_description)

            lres = feature_dict['lres']
            lres = tf.reshape(lres, [7, lres_size, lres_size])
            lres = tf.cast(lres, tf.float32)

            hres = feature_dict['hres']
            hres = tf.reshape(hres, [7, hres_size, hres_size])
            hres = tf.cast(hres, tf.float32)

            label = feature_dict['label']
            label = tf.reshape(label, [label_size, label_size, 1])
            return lres, hres, label

        @tf.function
        def _augment_function(lres_img, hres_img, label):
        # Transpose to [H, W, C]
            lres_img = tf.transpose(lres_img, [1, 2, 0])   # [60, 60, 12]
            hres_img = tf.transpose(hres_img, [1, 2, 0])   # [1000, 1000, 4]
    
            # Randomly choose 0, 90, 180, or 270 degrees
            k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
    
            # Apply rotation
            lres_img = tf.image.rot90(lres_img, k=k)
            hres_img = tf.image.rot90(hres_img, k=k)
            label = tf.image.rot90(label, k=k)
    
            # Transpose back to [C, H, W]
            lres_img = tf.transpose(lres_img, [2, 0, 1])   # [12, 60, 60]
            hres_img = tf.transpose(hres_img, [2, 0, 1])   # [4, 1000, 1000]
    
            return lres_img, hres_img, label

        dataset = tf.data.TFRecordDataset(filename)
        dataset = dataset.skip(skip)
        if take:
            dataset = dataset.take(take)
        if is_repeat:
            dataset = dataset.repeat()
        dataset = dataset.map(_parse_function)
        if is_train:
            dataset = dataset.map(_augment_function)
        if is_shuffle:
            dataset = dataset.shuffle(buffer_size=100)
        batch = dataset.batch(batch_size=batch_size)

        return batch
    # --------------------------------------------
    print('Begin Finetune')

    num_val = int(0.1 * num_training)   # 10% for validation
    num_train = num_training - num_val
    
    train_ds = input_pipeline_downstream_sr(
        finetune_tfrecords, 4, 0, num_train,
        is_shuffle=True, is_train=True, is_repeat=True
    )
    
    val_ds = input_pipeline_downstream_sr(
        finetune_tfrecords, 4, num_train, num_val,
        is_shuffle=False, is_train=False, is_repeat=False
    )
    
    best_val_loss = float('inf')
    
    for epoch in range(1):
    
        print(f'Epoch {epoch}')
        model.net_g.train()
    
        for step, (lr, hr, _) in enumerate(train_ds):
    
            lr = torch.from_numpy(lr.numpy().astype('float32')).to('cuda') * 0.0001
            hr = torch.from_numpy(hr.numpy().astype('float32')).to('cuda') * 0.0000275-0.2

            lr = (lr-0.5)/0.5
            hr = (hr-0.5)/0.5
    
            lr = F.interpolate(lr, size=(64, 64), mode='bilinear', align_corners=False)
            hr = F.interpolate(hr, size=(256, 256), mode='bilinear', align_corners=False)
    
            current_iter = step + 1
    
            model.update_learning_rate(
                current_iter,
                warmup_iter=opt['train'].get('warmup_iter', -1)
            )
    
            model.feed_data({'lq': lr, 'gt': hr}, training=False)
            model.optimize_parameters(current_iter)
    
            if step % 500 == 0:
    
                logs = model.get_current_log()
                train_loss = sum(logs.values())
    
                print(f'Step {step}, Train Loss: {train_loss:.4f}')
    
                # -------------------------
                # Validation
                # -------------------------
                model.net_g.eval()
    
                val_loss_total = 0
                val_steps = 0
    
                with torch.no_grad():
    
                    for val_lr, val_hr, _ in val_ds:
    
                        val_lr = torch.from_numpy(val_lr.numpy().astype('float32')).to('cuda') * 0.0001
                        val_hr = torch.from_numpy(val_hr.numpy().astype('float32')).to('cuda') * 0.0000275-0.2
    
                        val_lr = F.interpolate(val_lr, size=(64, 64), mode='bilinear', align_corners=False)
                        val_hr = F.interpolate(val_hr, size=(256, 256), mode='bilinear', align_corners=False)
    
                        model.feed_data({'lq': val_lr, 'gt': val_hr}, training=False)
                        model.test()
                        output = model.get_current_visuals()['result']
    
                        # logs = model.get_current_log()
                        val_loss = F.mse_loss(output.to('cuda'), val_hr)
    
                        val_loss_total += val_loss
                        val_steps += 1
    
                avg_val_loss = val_loss_total / val_steps
    
                print(f'Validation Loss: {avg_val_loss:.4f}')
    
                model.net_g.train()
    
                # -------------------------
                # Save best model
                # -------------------------
                if avg_val_loss < best_val_loss:
    
                    best_val_loss = avg_val_loss
                    print(f'New best model! Saving checkpoint (val={best_val_loss:.4f})')
    
                    ckpt = {
                        'iter': current_iter,
                        'val_loss': best_val_loss,
                        'net_g': model.get_bare_model(model.net_g).state_dict(),
                        'net_mse': model.get_bare_model(model.net_mse).state_dict(),
                        'opt_g': model.optimizer_g.state_dict(),
                    }
    
                    if hasattr(model, 'net_g_ema'):
                        ckpt['net_g_ema'] = model.get_bare_model(model.net_g_ema).state_dict()
    
                    if getattr(model, 'amp_scaler', None) is not None:
                        ckpt['amp_scaler'] = model.amp_scaler.state_dict()
    
                    torch.save(ckpt, finetuned_model)
    
            if step > 4000:
                break

    # --------------------------------------------

    ckpt = torch.load(finetuned_model, map_location='cuda')

    model.get_bare_model(model.net_g).load_state_dict(ckpt['net_g'], strict=True)
    model.get_bare_model(model.net_mse).load_state_dict(ckpt['net_mse'], strict=True)
    
    if 'opt_g' in ckpt:
        model.optimizer_g.load_state_dict(ckpt['opt_g'])
    
    if 'net_g_ema' in ckpt and hasattr(model, 'net_g_ema'):
        model.get_bare_model(model.net_g_ema).load_state_dict(ckpt['net_g_ema'], strict=True)
        
    print('Begin Write SR dataset')
    writer = tf.io.TFRecordWriter(output_tfrecords)

    def serialize_example(hres_4x, hres_ori, label):
        feature = {
            'hres_4x': tf.train.Feature(int64_list=tf.train.Int64List(value=hres_4x.reshape(-1))),
            'hres_ori': tf.train.Feature(int64_list=tf.train.Int64List(value=hres_ori.reshape(-1))),
            'label': tf.train.Feature(int64_list=tf.train.Int64List(value=label.reshape(-1)))
        }
        example_proto = tf.train.Example(features=tf.train.Features(feature=feature))
        return example_proto.SerializeToString()

    # Inference loop
    ds = input_pipeline_downstream_sr(finetune_tfrecords, 4, 0, num_sample, is_shuffle=False, is_train=False, is_repeat=False)
    for step, (lr, hr_ori, label) in enumerate(ds):
        lr = torch.from_numpy(lr.numpy().astype('float32')).to('cuda') * 0.0001
        hr = torch.from_numpy(hr_ori.numpy().astype('float32')).to('cuda') * 0.0000275-0.2
    
        lr = F.interpolate(lr, size=(64, 64), mode='bilinear', align_corners=False)
        hr = F.interpolate(hr, size=(256, 256), mode='bilinear', align_corners=False)
    
        model.feed_data({'lq': lr, 'gt': hr}, training=False)
        model.test()
        output = model.get_current_visuals()['result']
        output = output.detach().cpu().numpy()
        output = np.clip(output, 0, 1)
        output = ((output+0.2)/0.0000275).astype(int)
        # output[output < 0]=0
        label = label.numpy()

        if step < 4:
            lr = lr.detach().cpu().numpy()
        print('batch: ', step)

        for i in range(output.shape[0]):
            print(output[i].shape)
            print(hr_ori[i].numpy().shape)
            print(label[i].shape)
            example = serialize_example(output[i], hr_ori[i].numpy().astype('int'), label[i])
            writer.write(example)

            if step < 4:
                fig, axes = plt.subplots(1, 4, figsize=(16, 4))
                lr_show = np.transpose(lr[i], (1, 2, 0))
                axes[0].imshow(3*lr_show[:, :, [0,3,2]])
                axes[0].set_title('Low-Resolution')
                axes[0].axis('off')

                output_show = np.transpose(output[i], (1, 2, 0))
                axes[1].imshow(3*(output_show[:, :, 3:0:-1]*0.0000275-0.2))
                axes[1].set_title('High-Resolution')
                axes[1].axis('off')

                hr_show = np.transpose(hr_ori[i].numpy().astype('int'), (1, 2, 0))
                axes[2].imshow(3*(hr_show[:, :, 3:0:-1]*0.0000275-0.2))
                axes[2].set_title('High-Resolution')
                axes[2].axis('off')

                axes[3].imshow(label[i])
                axes[3].set_title('Output')
                axes[3].axis('off')

                plt.show()

    # Close writer
    writer.close()


In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 1065
num_training = 852
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_River.tfrecords']
finetuned_model = '/glade/derecho/scratch/lizhili/m2l8/UPSR_x4_M2L8_River_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_River_UPSR_x4.tfrecords'

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_model,
                output_tfrecords,
                model)

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 762
num_training = 610
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_CDL.tfrecords']
finetuned_model = '/glade/derecho/scratch/lizhili/m2l8/UPSR_x4_M2L8_CDL_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_CDL_UPSR_x4.tfrecords'

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_model,
                output_tfrecords,
                model)

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 256  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 1687
num_training = 1350
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_Urban.tfrecords']
finetuned_model = '/glade/derecho/scratch/lizhili/m2l8/UPSR_x4_M2L8_Urban_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_Urban_UPSR_x4.tfrecords'

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_model,
                output_tfrecords,
                model)

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 755
num_training = 604
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_GPP.tfrecords']
finetuned_model = '/glade/derecho/scratch/lizhili/m2l8/UPSR_x4_M2L8_GPP_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_GPP_UPSR_x4.tfrecords'

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_model,
                output_tfrecords,
                model)

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 256  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 1408
num_training = 1126
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_CHM.tfrecords']
finetuned_model = '/glade/derecho/scratch/lizhili/m2l8/UPSR_x4_M2L8_CHM_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_CHM_UPSR_x4.tfrecords'

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_model,
                output_tfrecords,
                model)